<a href="https://colab.research.google.com/github/FaridRash/brain-ct-hemorrhage-segmentation/blob/main/Notebooks/Segmentation/02_conversion_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Git Clone

In [ ]:
!git clone https://github.com/FaridRash/brain-ct-hemorrhage-segmentation
%cd brain-ct-hemorrhage-segmentation/Notebooks

Cloning into 'brain-ct-hemorrhage-segmentation'...
remote: Enumerating objects: 436, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 436 (delta 51), reused 45 (delta 23), pack-reused 350 (from 1)
Receiving objects: 100% (436/436), 578.59 MiB | 18.39 MiB/s, done.
Resolving deltas: 100% (177/177), done.


#Data Integrity Check

In [ ]:
import os
import nibabel as nib

ct_path = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/ct_scans"
mask_path = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/masks"

# get one sample file
ct_file = sorted(os.listdir(ct_path))[0]
mask_file = sorted(os.listdir(mask_path))[0]

print("CT file:", ct_file)
print("Mask file:", mask_file)

# load
ct_img = nib.load(os.path.join(ct_path, ct_file))
mask_img = nib.load(os.path.join(mask_path, mask_file))

# convert to numpy
ct_data = ct_img.get_fdata()
mask_data = mask_img.get_fdata()

# print info
print("\nCT shape:", ct_data.shape)
print("Mask shape:", mask_data.shape)

print("\nCT dtype:", ct_data.dtype)
print("Mask dtype:", mask_data.dtype)

print("\nCT spacing:", ct_img.header.get_zooms())
print("Mask spacing:", mask_img.header.get_zooms())

Shape: (512, 512, 39) → standard axial CT (39 slices)

Spacing:

x,y ≈ 0.41 mm (high resolution)

z = 5.0 mm (thick slices → low resolution in depth)

In [ ]:
import nibabel as nib

print("CT orientation:", nib.aff2axcodes(ct_img.affine))
print("Mask orientation:", nib.aff2axcodes(mask_img.affine))

Same orientation

No flipping issue

**The CT and mask are pixel-wise aligned**

#Visual Alignment Check

In [ ]:
import matplotlib.pyplot as plt

slice_idx = 20  # middle slice

plt.figure(figsize=(10,5))

# CT image
plt.subplot(1,2,1)
plt.imshow(ct_data[:, :, slice_idx], cmap='gray')
plt.title("CT Slice")
plt.axis("off")

# Mask
plt.subplot(1,2,2)
plt.imshow(mask_data[:, :, slice_idx], cmap='gray')
plt.title("Mask Slice")
plt.axis("off")

plt.show()

In [ ]:
plt.figure(figsize=(6,6))
plt.imshow(ct_data[:, :, slice_idx], cmap='gray')
plt.imshow(mask_data[:, :, slice_idx], cmap='jet', alpha=0.4)
plt.title("Overlay CT + Mask")
plt.axis("off")
plt.show()

In [ ]:
import numpy as np

print("Unique mask values:", np.unique(mask_data))

In [ ]:
def window_ct(img, level=40, width=80):
    min_val = level - width // 2
    max_val = level + width // 2
    img = np.clip(img, min_val, max_val)
    img = (img - min_val) / (max_val - min_val)
    return img

ct_windowed = window_ct(ct_data[:, :, 20])

plt.imshow(ct_windowed, cmap='gray')
plt.imshow(mask_data[:, :, 20], cmap='jet', alpha=0.4)
plt.axis("off")

In [ ]:
ct_slice = ct_data[:, :, 20]
mask_slice = mask_data[:, :, 20]

masked_pixels = ct_slice[mask_slice > 0]

print("Mean intensity inside mask:", masked_pixels.mean())
print("Mean intensity outside mask:", ct_slice[mask_slice == 0].mean())

#Conversion

##Train

In [ ]:
from google.colab import drive
import os

# mount drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd

train_split_path = "/content/drive/MyDrive/brain_ct_project/splits/train_scans.csv"

df_train = pd.read_csv(train_split_path)

train_scan_list = df_train["scan_id"].values

print("Train scans:", len(train_scan_list))
print("First 5 train IDs:", train_scan_list[:5])

In [ ]:
train_pairs = []

for scan_id in train_scan_list:

    scan_id = str(scan_id).zfill(3)

    ct_file = scan_id + ".nii"
    mask_file = scan_id + ".nii"

    ct_full_path = os.path.join(ct_path, ct_file)
    mask_full_path = os.path.join(mask_path, mask_file)

    if os.path.exists(ct_full_path) and os.path.exists(mask_full_path):
        train_pairs.append((ct_file, mask_file))
    else:
        print(f"WARNING: Missing file for scan {scan_id}")

print("Total matched train pairs:", len(train_pairs))

In [ ]:
import os

ct_path = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/ct_scans"
mask_path = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/masks"

train_pairs = []

for scan_id in train_scan_list:

    scan_id = str(scan_id).zfill(3)

    ct_file = scan_id + ".nii"
    mask_file = scan_id + ".nii"

    ct_full_path = os.path.join(ct_path, ct_file)
    mask_full_path = os.path.join(mask_path, mask_file)

    if os.path.exists(ct_full_path) and os.path.exists(mask_full_path):
        train_pairs.append((ct_file, mask_file))
    else:
        print(f"WARNING: Missing file for scan {scan_id}")

print("Total matched train pairs:", len(train_pairs))
print("First 5 train pairs:", train_pairs[:5])

In [ ]:
train_image_slices = []
train_mask_slices = []

for ct_file, mask_file in train_pairs:

    ct_full_path = os.path.join(ct_path, ct_file)
    mask_full_path = os.path.join(mask_path, mask_file)

    ct_vol = nib.load(ct_full_path).get_fdata()
    mask_vol = nib.load(mask_full_path).get_fdata()

    for i in range(ct_vol.shape[2]):

        img_slice = ct_vol[:, :, i]
        mask_slice = mask_vol[:, :, i]

        train_image_slices.append(img_slice)
        train_mask_slices.append(mask_slice)

print("Total train slices:", len(train_image_slices))

In [ ]:
import os

base_dir = "/content/processed_data"

train_dir = os.path.join(base_dir, "train")

train_img_npy_dir = os.path.join(train_dir, "images")
train_mask_npy_dir = os.path.join(train_dir, "masks")

for d in [train_img_npy_dir, train_mask_npy_dir]:
    os.makedirs(d, exist_ok=True)

print("Saving train to:", train_dir)

##Val

In [ ]:
import pandas as pd

val_split_path = "/content/drive/MyDrive/brain_ct_project/splits/val_scans.csv"

df_val = pd.read_csv(val_split_path)

val_scan_list = df_val["scan_id"].values

print("Val scans:", len(val_scan_list))
print("First 5 val IDs:", val_scan_list[:5])

In [ ]:
val_pairs = []

for scan_id in val_scan_list:

    scan_id = str(scan_id).zfill(3)

    ct_file = scan_id + ".nii"
    mask_file = scan_id + ".nii"

    ct_full_path = os.path.join(ct_path, ct_file)
    mask_full_path = os.path.join(mask_path, mask_file)

    if os.path.exists(ct_full_path) and os.path.exists(mask_full_path):
        val_pairs.append((ct_file, mask_file))
    else:
        print(f"WARNING: Missing file for scan {scan_id}")

print("Total matched val pairs:", len(val_pairs))

In [ ]:
import os

ct_path = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/ct_scans"
mask_path = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/masks"

val_pairs = []

for scan_id in val_scan_list:

    scan_id = str(scan_id).zfill(3)

    ct_file = scan_id + ".nii"
    mask_file = scan_id + ".nii"

    ct_full_path = os.path.join(ct_path, ct_file)
    mask_full_path = os.path.join(mask_path, mask_file)

    if os.path.exists(ct_full_path) and os.path.exists(mask_full_path):
        val_pairs.append((ct_file, mask_file))
    else:
        print(f"WARNING: Missing file for scan {scan_id}")

print("Total matched val pairs:", len(val_pairs))
print("First 5 val pairs:", val_pairs[:5])

In [ ]:
val_image_slices = []
val_mask_slices = []

for ct_file, mask_file in val_pairs:

    ct_full_path = os.path.join(ct_path, ct_file)
    mask_full_path = os.path.join(mask_path, mask_file)

    ct_vol = nib.load(ct_full_path).get_fdata()
    mask_vol = nib.load(mask_full_path).get_fdata()

    for i in range(ct_vol.shape[2]):

        img_slice = ct_vol[:, :, i]
        mask_slice = mask_vol[:, :, i]

        val_image_slices.append(img_slice)
        val_mask_slices.append(mask_slice)

print("Total val slices:", len(val_image_slices))

In [ ]:
import os

base_dir = "/content/processed_data"

val_dir = os.path.join(base_dir, "val")

val_img_npy_dir = os.path.join(val_dir, "images")
val_mask_npy_dir = os.path.join(val_dir, "masks")

for d in [val_img_npy_dir, val_mask_npy_dir]:
    os.makedirs(d, exist_ok=True)

print("Saving val to:", val_dir)

##Test

In [ ]:
import pandas as pd

test_split_path = "/content/drive/MyDrive/brain_ct_project/splits/test_scans.csv"

df_test = pd.read_csv(test_split_path)

test_scan_list = df_test["scan_id"].values

print("Test scans:", len(test_scan_list))
print("First 5 test IDs:", test_scan_list[:5])

In [ ]:
test_pairs = []

for scan_id in test_scan_list:

    scan_id = str(scan_id).zfill(3)

    ct_file = scan_id + ".nii"
    mask_file = scan_id + ".nii"

    ct_full_path = os.path.join(ct_path, ct_file)
    mask_full_path = os.path.join(mask_path, mask_file)

    if os.path.exists(ct_full_path) and os.path.exists(mask_full_path):
        test_pairs.append((ct_file, mask_file))
    else:
        print(f"WARNING: Missing file for scan {scan_id}")

print("Total matched test pairs:", len(test_pairs))

In [ ]:
import os

ct_path = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/ct_scans"
mask_path = "/content/brain-ct-hemorrhage-segmentation/Data/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/masks"

test_pairs = []

for scan_id in test_scan_list:

    scan_id = str(scan_id).zfill(3)

    ct_file = scan_id + ".nii"
    mask_file = scan_id + ".nii"

    ct_full_path = os.path.join(ct_path, ct_file)
    mask_full_path = os.path.join(mask_path, mask_file)

    if os.path.exists(ct_full_path) and os.path.exists(mask_full_path):
        test_pairs.append((ct_file, mask_file))
    else:
        print(f"WARNING: Missing file for scan {scan_id}")

print("Total matched test pairs:", len(test_pairs))
print("First 5 test pairs:", test_pairs[:5])

In [ ]:
test_image_slices = []
test_mask_slices = []

for ct_file, mask_file in test_pairs:

    ct_full_path = os.path.join(ct_path, ct_file)
    mask_full_path = os.path.join(mask_path, mask_file)

    ct_vol = nib.load(ct_full_path).get_fdata()
    mask_vol = nib.load(mask_full_path).get_fdata()

    for i in range(ct_vol.shape[2]):

        img_slice = ct_vol[:, :, i]
        mask_slice = mask_vol[:, :, i]

        test_image_slices.append(img_slice)
        test_mask_slices.append(mask_slice)

print("Total test slices:", len(test_image_slices))

In [ ]:
import os

base_dir = "/content/processed_data"

test_dir = os.path.join(base_dir, "test")

test_img_npy_dir = os.path.join(test_dir, "images")
test_mask_npy_dir = os.path.join(test_dir, "masks")

for d in [test_img_npy_dir, test_mask_npy_dir]:
    os.makedirs(d, exist_ok=True)

print("Saving test to:", test_dir)

#Save Data

##Save the files

###Train

In [ ]:
import pandas as pd

train_split_path = "/content/drive/MyDrive/brain_ct_project/splits/train_scans.csv"

df_train = pd.read_csv(train_split_path)
train_scan_list = df_train["scan_id"].astype(str).values

print("Using:", train_split_path)
print("Train scans:", len(train_scan_list))

In [ ]:
import os

train_pairs = []

for scan_id in train_scan_list:
    scan_id = str(scan_id).zfill(3)

    ct_file = scan_id + ".nii"
    mask_file = scan_id + ".nii"

    ct_full = os.path.join(ct_path, ct_file)
    mask_full = os.path.join(mask_path, mask_file)

    if os.path.exists(ct_full) and os.path.exists(mask_full):
        train_pairs.append((ct_file, mask_file))

print("Train pairs:", len(train_pairs))

In [ ]:
train_image_slices = []
train_mask_slices = []

for ct_file, mask_file in train_pairs:

    ct_full = os.path.join(ct_path, ct_file)
    mask_full = os.path.join(mask_path, mask_file)

    ct_vol = nib.load(ct_full).get_fdata()
    mask_vol = nib.load(mask_full).get_fdata()

    for i in range(ct_vol.shape[2]):
        train_image_slices.append(ct_vol[:, :, i])
        train_mask_slices.append(mask_vol[:, :, i])

print("Train slices:", len(train_image_slices))

In [ ]:
import shutil

shutil.rmtree("/content/processed_data/train", ignore_errors=True)

In [ ]:
base_dir = "/content/processed_data"

img_npy_dir = os.path.join(base_dir, "train", "images")
mask_npy_dir = os.path.join(base_dir, "train", "masks")

os.makedirs(img_npy_dir, exist_ok=True)
os.makedirs(mask_npy_dir, exist_ok=True)

print("Saving to:", img_npy_dir)

In [ ]:
import numpy as np

print("Saving:", len(train_image_slices))

for i, (img, mask) in enumerate(zip(train_image_slices, train_mask_slices)):

    np.save(os.path.join(train_img_npy_dir, f"slice_{i:05d}.npy"), img.astype(np.float32))
    np.save(os.path.join(train_mask_npy_dir, f"slice_{i:05d}.npy"), (mask > 0).astype(np.uint8))

print("TRAIN DONE")

In [ ]:
import os

print("Images:", len(os.listdir(train_img_npy_dir)))
print("Masks :", len(os.listdir(train_mask_npy_dir)))

In [ ]:
del train_image_slices
del train_mask_slices
del train_pairs
del df_train
del train_scan_list

import gc
gc.collect()

###Val

In [ ]:
import pandas as pd

val_split_path = "/content/drive/MyDrive/brain_ct_project/splits/val_scans.csv"

df_val = pd.read_csv(val_split_path)
val_scan_list = df_val["scan_id"].astype(str).values

print("Using:", val_split_path)
print("Val scans:", len(val_scan_list))

In [ ]:
import os

val_pairs = []

for scan_id in val_scan_list:
    scan_id = str(scan_id).zfill(3)

    ct_file = scan_id + ".nii"
    mask_file = scan_id + ".nii"

    ct_full = os.path.join(ct_path, ct_file)
    mask_full = os.path.join(mask_path, mask_file)

    if os.path.exists(ct_full) and os.path.exists(mask_full):
        val_pairs.append((ct_file, mask_file))

print("Val pairs:", len(val_pairs))

In [ ]:
val_image_slices = []
val_mask_slices = []

for ct_file, mask_file in val_pairs:

    ct_full = os.path.join(ct_path, ct_file)
    mask_full = os.path.join(mask_path, mask_file)

    ct_vol = nib.load(ct_full).get_fdata()
    mask_vol = nib.load(mask_full).get_fdata()

    for i in range(ct_vol.shape[2]):
        val_image_slices.append(ct_vol[:, :, i])
        val_mask_slices.append(mask_vol[:, :, i])

print("Val slices:", len(val_image_slices))

In [ ]:
import shutil

shutil.rmtree("/content/processed_data/val", ignore_errors=True)

In [ ]:
base_dir = "/content/processed_data"

val_img_npy_dir = os.path.join(base_dir, "val", "images")
val_mask_npy_dir = os.path.join(base_dir, "val", "masks")

os.makedirs(val_img_npy_dir, exist_ok=True)
os.makedirs(val_mask_npy_dir, exist_ok=True)

print("Saving to:", val_img_npy_dir)

In [ ]:
import numpy as np

print("Saving:", len(val_image_slices))

for i, (img, mask) in enumerate(zip(val_image_slices, val_mask_slices)):

    np.save(os.path.join(val_img_npy_dir, f"slice_{i:05d}.npy"), img.astype(np.float32))
    np.save(os.path.join(val_mask_npy_dir, f"slice_{i:05d}.npy"), (mask > 0).astype(np.uint8))

print("VAL DONE")

In [ ]:
import os

print("VAL Images:", len(os.listdir(val_img_npy_dir)))
print("VAL Masks :", len(os.listdir(val_mask_npy_dir)))

In [ ]:
del val_image_slices
del val_mask_slices
del val_pairs
del df_val
del val_scan_list

gc.collect()

###Test

In [ ]:
import pandas as pd

test_split_path = "/content/drive/MyDrive/brain_ct_project/splits/test_scans.csv"

df_test = pd.read_csv(test_split_path)
test_scan_list = df_test["scan_id"].astype(str).values

print("Using:", test_split_path)
print("Test scans:", len(test_scan_list))

In [ ]:
import os

test_pairs = []

for scan_id in test_scan_list:
    scan_id = str(scan_id).zfill(3)

    ct_file = scan_id + ".nii"
    mask_file = scan_id + ".nii"

    ct_full = os.path.join(ct_path, ct_file)
    mask_full = os.path.join(mask_path, mask_file)

    if os.path.exists(ct_full) and os.path.exists(mask_full):
        test_pairs.append((ct_file, mask_file))

print("Test pairs:", len(test_pairs))

In [ ]:
test_image_slices = []
test_mask_slices = []

for ct_file, mask_file in test_pairs:

    ct_full = os.path.join(ct_path, ct_file)
    mask_full = os.path.join(mask_path, mask_file)

    ct_vol = nib.load(ct_full).get_fdata()
    mask_vol = nib.load(mask_full).get_fdata()

    for i in range(ct_vol.shape[2]):
        test_image_slices.append(ct_vol[:, :, i])
        test_mask_slices.append(mask_vol[:, :, i])

print("Test slices:", len(test_image_slices))

In [ ]:
import shutil

shutil.rmtree("/content/processed_data/test", ignore_errors=True)

In [ ]:
base_dir = "/content/processed_data"

test_img_npy_dir = os.path.join(base_dir, "test", "images")
test_mask_npy_dir = os.path.join(base_dir, "test", "masks")

os.makedirs(test_img_npy_dir, exist_ok=True)
os.makedirs(test_mask_npy_dir, exist_ok=True)

print("Saving to:", test_img_npy_dir)

In [ ]:
import numpy as np

print("Saving:", len(test_image_slices))

for i, (img, mask) in enumerate(zip(test_image_slices, test_mask_slices)):

    np.save(os.path.join(test_img_npy_dir, f"slice_{i:05d}.npy"), img.astype(np.float32))
    np.save(os.path.join(test_mask_npy_dir, f"slice_{i:05d}.npy"), (mask > 0).astype(np.uint8))

print("TEST DONE")

In [ ]:
import os

print("TEST Images:", len(os.listdir(test_img_npy_dir)))
print("TEST Masks :", len(os.listdir(test_mask_npy_dir)))

In [ ]:
del test_image_slices
del test_mask_slices
del test_pairs
del df_test
del test_scan_list

gc.collect()

##Test the data

In [ ]:
import os

for split in ["train", "val", "test"]:
    path = f"/content/processed_data/{split}/images"
    if os.path.exists(path):
        print(split, len(os.listdir(path)))
    else:
        print(split, "NOT CREATED")

#Blood Range Fixing

In [ ]:
import os
import numpy as np
from tqdm import tqdm

base_dir = "/content/drive/MyDrive/brain_ct_project/processed_data"

splits = ["train", "val", "test"]

LOW = 20     # conservative lower bound
HIGH = 120   # generous upper bound

for split in splits:
    img_dir = os.path.join(base_dir, split, "images")
    mask_dir = os.path.join(base_dir, split, "masks")

    files = sorted(os.listdir(img_dir))

    for f in tqdm(files, desc=split):
        img = np.load(os.path.join(img_dir, f))
        mask = np.load(os.path.join(mask_dir, f))

        # --- normalize shapes ---
        if img.ndim == 3:
            img = img[0]
        if mask.ndim == 3:
            mask = mask[0]

        # --- CLEANING ---
        valid_region = (img >= LOW) & (img <= HIGH)
        clean_mask = (mask > 0) & valid_region

        # convert back to uint8 (0/1)
        clean_mask = clean_mask.astype(np.uint8)

        # --- SAVE BACK ---
        np.save(os.path.join(mask_dir, f), clean_mask)

#Export zipped files to Google.Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil
import os

# source folder
source_dir = "/content/processed_data"

# destination (Drive)
save_dir = "/content/drive/MyDrive/brain_ct_project"
os.makedirs(save_dir, exist_ok=True)

zip_path = os.path.join(save_dir, "processed_data_full.zip")

# create zip
shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=source_dir
)

print("ZIP saved to:", zip_path)

ZIP saved to: /content/drive/MyDrive/brain_ct_project/processed_data_full.zip
